# 어텐션을 통한 역전파: 단계별 - 어텐션 역전파의 수학

- Tutorial ID: `ull-3`
- Tutorial: 어텐션을 통한 역전파: 단계별
- Section ID: `ull-3-1`
- Section: 어텐션 역전파의 수학


In [ ]:
# ============================================================
# 코드 읽는 법 — 어텐션 역전파의 수학
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 62)
print("어텐션 역전파: 단계별 추적")
print("=" * 62)

def softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

np.random.seed(42)

# 작은 크기로 설정
seq_len = 2
d_model = 4
d_head = 2

X = np.random.randn(seq_len, d_model) * 0.5
W_Q = np.random.randn(d_model, d_head) * 0.3
W_K = np.random.randn(d_model, d_head) * 0.3
W_V = np.random.randn(d_model, d_head) * 0.3
W_O = np.random.randn(d_head, d_model) * 0.3

In [ ]:
# 순전파 (Forward Pass)

In [ ]:
print("\n--- 순전파 ---")
scale = np.sqrt(d_head)

Q = X @ W_Q;           print(f"1. Q = X @ W_Q:       shape {Q.shape}")
K = X @ W_K;           print(f"2. K = X @ W_K:       shape {K.shape}")
V = X @ W_V;           print(f"3. V = X @ W_V:       shape {V.shape}")
S = Q @ K.T / scale;   print(f"4. S = Q @ K^T / √d:  shape {S.shape}")

# 인과적 마스크
mask = np.triu(np.full((seq_len, seq_len), -1e9), k=1)
S_masked = S + mask

A = softmax(S_masked);  print(f"5. A = softmax(S):    shape {A.shape}")
O = A @ V;              print(f"6. O = A @ V:         shape {O.shape}")
Y = O @ W_O;            print(f"7. Y = O @ W_O:       shape {Y.shape}")

# 간단한 손실: L = sum(Y^2) / 2
L = np.sum(Y ** 2) / 2
print(f"\n   Loss = ||Y||^2 / 2 = {L:.6f}")

In [ ]:
# 역전파 (Backward Pass)

In [ ]:
print("\n--- 역전파 ---")

# 7. ∂L/∂Y = Y (MSE 미분)
dL_dY = Y.copy()
print(f"7. dL/dY = Y:         shape {dL_dY.shape}")

# dL/dW_O = O^T @ dL/dY
dL_dW_O = O.T @ dL_dY
print(f"   dL/dW_O:           shape {dL_dW_O.shape}")

# 6. dL/dO = dL/dY @ W_O^T
dL_dO = dL_dY @ W_O.T
print(f"6. dL/dO = dL/dY @ W_O^T:  shape {dL_dO.shape}")

# dL/dA = dL/dO @ V^T
dL_dA = dL_dO @ V.T
print(f"   dL/dA = dL/dO @ V^T:    shape {dL_dA.shape}")

# dL/dV = A^T @ dL/dO
dL_dV = A.T @ dL_dO
print(f"   dL/dV = A^T @ dL/dO:    shape {dL_dV.shape}")

# 5. 소프트맥스 역전파
# dL/dS[i,j] = sum_k dL/dA[i,k] * A[i,k] * (delta(j,k) - A[i,j])
# = A[i,j] * (dL/dA[i,j] - sum_k dL/dA[i,k] * A[i,k])
dL_dS = A * (dL_dA - np.sum(dL_dA * A, axis=-1, keepdims=True))
# 마스킹된 위치는 그래디언트 0
dL_dS = np.where(mask == 0, dL_dS, 0)
print(f"5. dL/dS (softmax grad): shape {dL_dS.shape}")

# 4. dL/dQ = dL/dS @ K / √d
dL_dQ = dL_dS @ K / scale
print(f"4. dL/dQ = dL/dS @ K / √d: shape {dL_dQ.shape}")

# dL/dK = dL/dS^T @ Q / √d
dL_dK = dL_dS.T @ Q / scale
print(f"   dL/dK = dL/dS^T @ Q / √d: shape {dL_dK.shape}")

# 1-3. 가중치 그래디언트
dL_dW_Q = X.T @ dL_dQ
dL_dW_K = X.T @ dL_dK
dL_dW_V = X.T @ dL_dV

print(f"1. dL/dW_Q:           shape {dL_dW_Q.shape}")
print(f"2. dL/dW_K:           shape {dL_dW_K.shape}")
print(f"3. dL/dW_V:           shape {dL_dW_V.shape}")

In [ ]:
# 수치적 검증 (Numerical Gradient Check)

In [ ]:
print("\n--- 수치적 검증 (유한 차분법) ---")

eps = 1e-5

def compute_loss(W_Q_, W_K_, W_V_, W_O_):
    Q_ = X @ W_Q_
    K_ = X @ W_K_
    V_ = X @ W_V_
    S_ = Q_ @ K_.T / scale + mask
    A_ = softmax(S_)
    O_ = A_ @ V_
    Y_ = O_ @ W_O_
    return np.sum(Y_ ** 2) / 2

# W_Q의 [0,0] 원소 검증
W_Q_plus = W_Q.copy(); W_Q_plus[0, 0] += eps
W_Q_minus = W_Q.copy(); W_Q_minus[0, 0] -= eps
numerical_grad = (compute_loss(W_Q_plus, W_K, W_V, W_O) - compute_loss(W_Q_minus, W_K, W_V, W_O)) / (2 * eps)
analytical_grad = dL_dW_Q[0, 0]

print(f"  dL/dW_Q[0,0]:")
print(f"    해석적 그래디언트: {analytical_grad:.8f}")
print(f"    수치적 그래디언트: {numerical_grad:.8f}")
print(f"    상대 오차: {abs(analytical_grad - numerical_grad) / (abs(analytical_grad) + 1e-10):.2e}")

# W_O 검증
W_O_plus = W_O.copy(); W_O_plus[0, 0] += eps
W_O_minus = W_O.copy(); W_O_minus[0, 0] -= eps
num_grad_O = (compute_loss(W_Q, W_K, W_V, W_O_plus) - compute_loss(W_Q, W_K, W_V, W_O_minus)) / (2 * eps)

print(f"\n  dL/dW_O[0,0]:")
print(f"    해석적: {dL_dW_O[0,0]:.8f}")
print(f"    수치적: {num_grad_O:.8f}")
print(f"    상대 오차: {abs(dL_dW_O[0,0] - num_grad_O) / (abs(dL_dW_O[0,0]) + 1e-10):.2e}")

print("""
요약:
  - 어텐션 역전파는 순전파의 역순으로 체인 룰 적용
  - 소프트맥스 역전파가 가장 복잡 (야코비안 필요)
  - 수치적 그래디언트 체크로 구현 정확성 검증
  - 실제 프레임워크(PyTorch)는 이를 자동으로 수행!
""")